# HW2 Question 1 - 1D Convolution

The build cell asks for `conv1d.cu` if it is not in `/content` yet.

In [ ]:
!nvidia-smi --query-gpu=name,compute_cap,driver_version --format=csv
!nvcc --version | tail -2
!ncu --version | tail -1

In [ ]:
%cd /content
import os
if not os.path.exists('conv1d.cu'):
    from google.colab import files
    files.upload()  # choose conv1d.cu
!ls -l conv1d.cu
!mkdir -p results
# TILE_WIDTH is a compile-time constant (as in the skeleton): build one binary per tile size
!for t in 32 64 128 256 512 1024; do nvcc -O3 -arch=sm_75 -lineinfo -DTILE_WIDTH=$t conv1d.cu -o conv1d_T$t || break; done; ls conv1d_T*

## Q1.3 Run with N = 1024 ... 16384

Output for the report screenshot, using the chosen tile size TILE_WIDTH = 128.

In [ ]:
%%bash
cd /content
for n in 1024 2048 4096 8192 16384; do
  ./conv1d_T128 $n | tee results/run_$n.txt
  echo
done

## Q1.3 Tile size sweep

Kernel time for every N and tile size. Run it twice to see the run-to-run noise.

In [ ]:
## Q1.4 Nsight Systems

Whole-program timeline: memory copies, kernel launches and CUDA API time, with the chosen tile size 128.

In [ ]:
%%bash
cd /content
NSYS=$(which nsys || ls /usr/local/cuda*/bin/nsys /opt/nvidia/nsight-systems/*/bin/nsys 2>/dev/null | head -1)
echo "Using $NSYS"
$NSYS profile --force-overwrite true -o results/q1_nsys ./conv1d 16384 256 > /dev/null
$NSYS stats --report cuda_api_sum,cuda_gpu_kern_sum,cuda_gpu_mem_time_sum results/q1_nsys.nsys-rep | tee results/nsys_stats.txt

# --launch-skip 1 skips the warm-up launch
for k in convolution_1D_tiled convolution_1D_basic; do
  ncu --kernel-name $k --launch-skip 1 --launch-count 1 $SECTIONS ./conv1d_T128 16384 2>&1 | tee results/ncu_$k.txt
done
# full report for the GUI
ncu --kernel-name convolution_1D_tiled --launch-skip 1 --launch-count 1 --set full -f -o results/q1_tiled_full ./conv1d_T128 16384 > /dev/null

In [ ]:
## Q1.4 Shared memory capacity

The second argument sets `cudaFuncAttributePreferredSharedMemoryCarveout` for the tiled kernel (percentage of the unified L1/shared storage used as shared memory). Compare kernel time and the `Shared Memory Configuration Size` / occupancy reported by ncu.

In [ ]:
%%bash
cd /content
SECTIONS="--section SpeedOfLight --section Occupancy --section LaunchStats"
KEYS="Shared Memory Configuration Size|Achieved Occupancy|Theoretical Occupancy|Block Limit Shared Mem|Duration"
for c in 0 25 50 100; do
  echo "== carveout $c%"
  ./conv1d 16384 256 $c | tail -5
  ncu --kernel-name conv1d_tiled --launch-skip 1 --launch-count 1 $SECTIONS ./conv1d 16384 256 $c 2>&1 | grep -E "$KEYS"
done | tee results/carveout.txt

In [ ]:
!cd /content && zip -r q1_results.zip results
from google.colab import files
files.download('/content/q1_results.zip')